# Map-reduce

## Review

We're building up to a multi-agent research assistant that ties together all of the modules from this course.

To build this multi-agent assistant, we've been introducing a few LangGraph controllability topics.

We just covered parallelization and sub-graphs.

## Goals

Now, we're going to cover [map reduce](https://docs.langchain.com/oss/python/langgraph/use-graph-api#map-reduce-and-the-send-api).

> ### 📌 Reducers — read this once, applies to every notebook in this module
>
> **One-line definition:** *A reducer controls how a new value is merged into the existing value for a key. You need one when you either want accumulation, or when multiple nodes write the same key concurrently and you can't afford to lose any write.*
>
> You attach one with `Annotated[<type>, <reducer_fn>]` inside your `TypedDict`.
>
> **The two triggers in plain terms:**
>
> 1. **Accumulation** — the value should grow over the run (a list of messages, context docs, section drafts), not be replaced each step. Pick `add_messages`, `operator.add`, or a custom merge/sort reducer.
> 2. **Concurrent writes** — fan-out, parallel branches, or the `Send()` API have multiple nodes writing the same key in the **same step**. Without a reducer LangGraph raises `INVALID_CONCURRENT_GRAPH_UPDATE` ("Can receive only one value per step…") because it doesn't know which concurrent write to keep — and you would lose every write but one.
>
> **Default behavior with no reducer:**
> - Sequential writes → **last write wins** (overwrite).
> - Two writes in the same step → **error** (not a silent overwrite).
>
> **In this notebook specifically:** `Send("generate_joke", {"subject": s})` dispatches **one parallel `generate_joke` execution per subject**, and every one of them writes back to the *same* `jokes` key in the same super-step. That is the canonical map-reduce shape, and that is exactly why the schema declares `jokes: Annotated[list, operator.add]`. Drop that `operator.add` and the run will hard-error on the very first batch — no joke would survive.
>
> **On the original phrasing.** The first attempt — *"Reducers are being used when the state is important to be appended, or the key is being written by many nodes at one time, which reducers solve the overwrite issue in this case"* — is *directionally* right but slightly off in two places:
> - "the state is important to be appended" → it's the value at a *key*, not the whole state. The correct trigger is simply: **the value should accumulate**.
> - "reducers solve the overwrite issue" → in the concurrent case there is no silent overwrite to solve; LangGraph **hard-errors** instead. The reducer is what turns that error into a defined merge so no write is dropped.

In [60]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph

In [61]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("MISTRAL_API_KEY")

We'll use [LangSmith](https://docs.langchain.com/langsmith/home) for [tracing](https://docs.langchain.com/langsmith/observability-concepts).

In [62]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## Problem

Map-reduce operations are essential for efficient task decomposition and parallel processing. 

It has two phases:

(1) `Map` - Break a task into smaller sub-tasks, processing each sub-task in parallel.

(2) `Reduce` - Aggregate the results across all of the completed, parallelized sub-tasks.

Let's design a system that will do two things:

(1) `Map` - Create a set of jokes about a topic.

(2) `Reduce` - Pick the best joke from the list.

We'll use an LLM to do the job generation and selection.

In [63]:
from langchain_mistralai import ChatMistralAI

# Prompts we will use
subjects_prompt = """Generate a list of 3 sub-topics that are all related to this overall topic: {topic}."""
joke_prompt = """Generate a joke about {subject}"""
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return the ID of the best one, starting 0 as the ID for the first joke. Jokes: \n\n  {jokes}"""

# LLM
model = ChatMistralAI(model="mistral-medium", temperature=0) 

## State

### Parallelizing joke generation

First, let's define the entry point of the graph that will:

* Take a user input topic
* Produce a list of joke topics from it
* Send each joke topic to our above joke generation node

Our state has a `jokes` key, which will accumulate jokes from parallelized joke generation

In [64]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel

class Subjects(BaseModel):
    subjects: list[str]

class BestJoke(BaseModel):
    id: int
    
class OverallState(TypedDict):
    topic: str
    subjects: list
    jokes: Annotated[list, operator.add]
    best_selected_joke: str

Generate subjects for jokes.

In [65]:
def generate_topics(state: OverallState):
    prompt = subjects_prompt.format(topic = state["topic"])
    response = model.with_structured_output(Subjects).invoke(prompt)
    return {"subjects": response.subjects}

Here is the magic: we use the  [Send](https://docs.langchain.com/oss/python/langgraph/graph-api/#send) to create a joke for each subject.

This is very useful! It can automatically parallelize joke generation for any number of subjects.

* `generate_joke`: the name of the node in the graph
* `{"subject": s`}: the state to send

`Send` allow you to pass any state that you want to `generate_joke`! It does not have to align with `OverallState`.

In this case, `generate_joke` is using its own internal state, and we can populate this via `Send`.

In [66]:
from langgraph.types import Send
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

### Joke generation (map)

Now, we just define a node that will create our jokes, `generate_joke`!

We write them back out to `jokes` in `OverallState`! 

This key has a reducer that will combine lists.

In [67]:
class JokeState(TypedDict):
    subject: str 

class Joke(BaseModel): 
    joke: str

def generate_joke(state: JokeState):
    prompt = joke_prompt.format(subject = state["subject"])
    response = model.with_structured_output(Joke).invoke(prompt)
    return {"jokes": [response.joke]}

### Best joke selection (reduce)

Now, we add logic to pick the best joke.

In [ ]:
def best_joke(state: OverallState):
    jokes = "\n\n".join(state["jokes"])
    prompt = best_joke_prompt.format(topic=state["topic"], jokes=jokes)
    response = model.with_structured_output(BestJoke).invoke(prompt)
    return {"best_selected_joke": state["jokes"][response.id]}

## Compile

In [ ]:
from IPython.display import Image
from langgraph.graph import END, StateGraph, START

# Construct the graph: here we put everything together to construct our graph
graph = StateGraph(OverallState)

# Add nodes
graph.add_node("generate_topics", generate_topics)
graph.add_node("generate_joke", generate_joke)
graph.add_node("best_joke", best_joke)

# Add edges
graph.add_edge(START, "generate_topics")
graph.add_conditional_edges("generate_topics", continue_to_jokes, ["generate_joke"])
graph.add_edge("generate_joke", "best_joke")
graph.add_edge("best_joke", END)

# Compile the graph
app = graph.compile()
Image(app.get_graph().draw_mermaid_png())

In [70]:
# Call the graph: here we call it to generate a list of jokes
for s in app.stream({"topic": "animals"}):
    print(s)

{'generate_topics': {'subjects': ['animal behavior', 'endangered species and conservation', 'animal adaptations and evolution']}}
{'generate_joke': {'jokes': ['Why did the endangered panda refuse to leave the conservation area? Because it didn’t want to become *extinct* from the group chat!']}}
{'generate_joke': {'jokes': ["Why did the chameleon refuse to play hide and seek with the other animals? Because it was tired of always blending in and never being *seen* for its true colors—evolution’s way of saying, 'Adapt or get left out!'"]}}
{'generate_joke': {'jokes': ["Why don't skeletons fight each other? They don't have the guts. But you know who does? Cats—they’ll stare at a wall for hours like it owes them money, and then suddenly sprint across the house at 3 AM like they’re training for the Olympics. Animal behavior: nature’s original comedy show!"]}}
{'best_joke': {'best_selected_joke': "Why don't skeletons fight each other? They don't have the guts. But you know who does? Cats—they